# IQL Sweep Results Table

Collects results from W&B sweeps and formats them into the paper-style table
(rows = datasets, columns = methods).

For every **run** in a sweep we take the **max** of the eval `mean_score`
metric over training, then report the **mean ± error** of those per-run maxima
across the sweep (one run per seed).

The sweeps listed below populate the **"IQL with task reward"** column. Add
more sweep IDs to `SWEEPS` as new results come in and re-run the notebook.

> **Scaling note.** `iql.py` logs `mean_score` as the raw evaluation return.
> For antmaze the per-episode reward is sparse (1 on reaching the goal, else 0),
> so `mean_score` is the success rate in `[0, 1]`. We multiply by `SCALE = 100`
> to match the 0–100 scale shown in the paper table. Change `SCALE` if your
> metric is already normalized.

In [ ]:
# !pip install wandb pandas numpy  # uncomment if needed
import time

import numpy as np
import pandas as pd
import wandb

api = wandb.Api()

## Configuration

- `ENTITY`: your W&B entity (team/user). Leave as `""` to use your default
  entity from `wandb login`.
- `PROJECT`: the W&B project the sweeps live in.
- `METRIC`: the logged metric to maximize per run.
- `SCALE`: multiplier applied to the metric (100 turns an antmaze success rate
  into a 0–100 score).
- `ERROR_TYPE`: what the `±` shows — `"std"`, `"var"`, or `"sem"`.
- `DDOF`: delta-degrees-of-freedom for std/var (`1` = sample std, `0` =
  population std).

In [ ]:
ENTITY = "champlin-university-of-arizona"  # "" = default entity from wandb login
PROJECT = "IQL-pref"
METRIC = "mean_score"
SCALE = 100.0
ERROR_TYPE = "std"  # "std" | "var" | "sem"
DDOF = 1  # 1 = sample std/var (recommended across seeds)

## Sweep registry

`SWEEPS[dataset][method] = sweep_id`. Method keys map to table columns via
`METHOD_COLUMNS` below. Fill in more IDs as you get results — empty entries
render as blank cells.

In [ ]:
SWEEPS = {
    "antmaze-medium-play-v2": {
        "task_reward": "567akajk",
    },
    "antmaze-medium-diverse-v2": {
        "task_reward": "new66902",
    },
    "antmaze-large-play-v2": {
        # "task_reward": "...",
    },
    "antmaze-large-diverse-v2": {
        # "task_reward": "...",
    },
}

# (method_key, (column group, column subheader)) — controls table layout/order.
METHOD_COLUMNS = [
    ("task_reward", ("IQL with task reward", "")),
    ("MR", ("IQL with preference learning", "MR")),
    ("NMR", ("IQL with preference learning", "NMR")),
    ("PT", ("IQL with preference learning", "PT (ours)")),
]

## Fetch + aggregate

`fetch_run_max_scores` pulls the `mean_score` series for every run in a sweep
and returns the per-run maxima, already scaled.

It uses `run.history(...)` (the **sampled-history** endpoint — one request per
run) rather than `run.scan_history(...)`, which paginates through *every*
logged training step and is much slower. `samples=100000` is far above the
number of eval points, so the max is exact. Per-run progress is printed so a
slow sweep is visibly progressing, not hung. Results are cached by `sweep_id`;
set `force=True` to refetch.

In [ ]:
_cache = {}


def _run_max(run, metric=METRIC):
    """Max of `metric` for one run via the fast sampled-history endpoint."""
    df = run.history(keys=[metric], samples=100000, pandas=True)
    if df is None or metric not in getattr(df, "columns", []):
        return None
    series = df[metric].dropna()
    return float(series.max()) if not series.empty else None


def fetch_run_max_scores(
    sweep_id, metric=METRIC, scale=SCALE, force=False, verbose=True
):
    """Return a list of per-run max(metric) values (scaled) for one sweep."""
    if sweep_id in _cache and not force:
        return _cache[sweep_id]
    path = f"{ENTITY}/{PROJECT}/{sweep_id}" if ENTITY else f"{PROJECT}/{sweep_id}"
    sweep = api.sweep(path)
    runs = list(sweep.runs)
    if verbose:
        print(f"  sweep {sweep_id}: {len(runs)} run(s)")
    maxima, t0 = [], time.time()
    for i, run in enumerate(runs, 1):
        m = _run_max(run, metric)
        if m is not None:
            maxima.append(scale * m)
        if verbose:
            shown = f"{scale * m:.2f}" if m is not None else "no data"
            print(f"    [{i}/{len(runs)}] {run.name} ({run.state}): {shown}")
    if verbose:
        print(f"  -> {len(maxima)} run(s) with data in {time.time() - t0:.1f}s")
    _cache[sweep_id] = maxima
    return maxima


def summarize(maxima, ddof=DDOF):
    arr = np.asarray(maxima, dtype=float)
    n = arr.size
    if n == 0:
        return None
    std = float(arr.std(ddof=ddof)) if n > ddof else 0.0
    var = float(arr.var(ddof=ddof)) if n > ddof else 0.0
    return {
        "mean": float(arr.mean()),
        "std": std,
        "var": var,
        "sem": std / np.sqrt(n) if n else 0.0,
        "n": n,
        "values": arr,
    }


def fmt(stats, err=ERROR_TYPE):
    if stats is None or stats["n"] == 0:
        return ""
    return f"{stats['mean']:.2f} ± {stats[err]:.2f}"

In [ ]:
# Pull everything once into a {dataset: {method: stats}} structure.
results = {}
for dataset, methods in SWEEPS.items():
    results[dataset] = {}
    for method, sweep_id in methods.items():
        if not sweep_id:
            continue
        print(f"{dataset}  [{method}]")
        maxima = fetch_run_max_scores(sweep_id)
        results[dataset][method] = summarize(maxima)
        print(f"  => {fmt(results[dataset][method])}\n")

## Results table

In [ ]:
columns = pd.MultiIndex.from_tuples([col for _, col in METHOD_COLUMNS])
rows = list(SWEEPS.keys())

table = pd.DataFrame(index=rows, columns=columns, dtype=object)
table.index.name = "Dataset"
for dataset in rows:
    for method, col in METHOD_COLUMNS:
        table.loc[dataset, col] = fmt(results.get(dataset, {}).get(method))

table.fillna("", inplace=True)
table

## Per-run detail

Inspect the individual per-run maxima behind each populated cell.

In [ ]:
detail_rows = []
for dataset, methods in SWEEPS.items():
    for method, sweep_id in methods.items():
        stats = results.get(dataset, {}).get(method)
        if not stats:
            continue
        detail_rows.append(
            {
                "dataset": dataset,
                "method": method,
                "sweep_id": sweep_id,
                "n_runs": stats["n"],
                "mean": round(stats["mean"], 2),
                "std": round(stats["std"], 2),
                "var": round(stats["var"], 2),
                "sem": round(stats["sem"], 2),
                "values": np.round(np.sort(stats["values"])[::-1], 2).tolist(),
            }
        )
pd.DataFrame(detail_rows)

## Export

Copy-pasteable Markdown and LaTeX versions of the table.

In [ ]:
print("### Markdown\n")
print(table.to_markdown())
print("\n### LaTeX\n")
print(table.to_latex(multicolumn=True, multicolumn_format="c"))
# table.to_csv("results_table.csv")  # uncomment to save